# Implementación de Interpolación y Regresión Lineal bajo CRISP-DM
### Skyfusion Analytics · Cuenca del Río Combeima (Ibagué, Tolima)
**Curso:** Minería de Datos · UNIMINUTO Ibagué · 2026-2  
**Metodología:** CRISP-DM (Cross-Industry Standard Process for Data Mining)

---

```
┌────────────────────────────────────────────────────────┐
│                   FASES CRISP-DM                      │
├──────────────┬─────────────────────────────────────────┤
│ Fase 1       │ Comprensión del Negocio                 │
│ Fase 2       │ Comprensión de los Datos                │
│ Fase 3       │ Preparación de los Datos (Interpolación)│
│ Fase 4       │ Modelado (Regresión Lineal Simple y Múltiple) │
│ Fase 5       │ Evaluación y Diagnóstico de Residuos    │
│ Fase 6       │ Despliegue y Acciones Multiagente       │
└──────────────┴─────────────────────────────────────────┘
```

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Importar herramientas especializadas de Skyfusion
sys.path.insert(0, str(Path('../skills/ml_tools').resolve()))
from spectral_interpolator import SpectralTemporalInterpolator
from crisp_linear_regression import CrispLinearRegressor


## Fase 1: Comprensión del Negocio (Business Understanding)
- **Problema:** En la cuenca del río Combeima, la presencia de nubosidad recurrente interrumpe las series satelitales ópticas.
- **Objetivo Técnico:**
  1. Reconstruir la continuidad temporal de la señal espectral con técnicas de interpolación matemática.
  2. Entrenar modelos de regresión lineal para aislar la tasa de degradación/regeneración multianual y las relaciones biofísicas entre índices.

## Fase 2: Comprensión de los Datos (Data Understanding)
Carga y exploración del dataset multitemporal GEE de la cuenca.

In [ ]:
data_path = Path('../rnn_combeima/data/raw/series_multisensor_combeima.csv')
df = pd.read_csv(data_path)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

print(f"Total observaciones: {len(df)}")
print(f"Rango temporal: {df['fecha'].min().date()} a {df['fecha'].max().date()}")
display(df.describe().T)


## Fase 3: Preparación de los Datos (Data Preparation) — Interpolación
Comparamos métodos de interpolación mediante simulación de huecos artificiales (20% enmascarado).

In [ ]:
interpolator = SpectralTemporalInterpolator(date_col='fecha')
serie_media = df.groupby('fecha')[['NDVI', 'EVI']].mean().reset_index()

eval_interp = interpolator.evaluate_methods(serie_media['NDVI'], gap_percentage=0.20, gap_block_size=3)
display(eval_interp)

# Serie completamente interpolada por tiempo
serie_interp = interpolator.interpolate_dataframe(serie_media, value_cols=['NDVI', 'EVI'], method='time')

plt.figure(figsize=(12, 4))
plt.plot(serie_media['fecha'], serie_media['NDVI'], 'o', alpha=0.5, label='Observaciones Originales')
plt.plot(serie_interp['fecha'], serie_interp['NDVI'], '-', color='#10b981', label='Interpolación Temporal Continua')
plt.title('Reconstrucción de Serie Temporal NDVI (Combeima)')
plt.ylabel('NDVI')
plt.legend()
plt.show()


## Fase 4: Modelado (Modeling) — Regresión Lineal
Ajuste de modelos paramétricos:
1. **Regresión Lineal Simple:** $NDVI = \beta_0 + \beta_1 \cdot t$ (Tendencia temporal pura)
2. **Regresión Lineal Múltiple:** $NDVI = \beta_0 + \beta_1 \cdot t + \beta_2 \cdot EVI + \beta_3 \cdot \sin(\omega t) + \beta_4 \cdot \cos(\omega t)$

In [ ]:
# Ingeniería de variables temporales y estacionales
serie_interp['dia_juliano'] = serie_interp['fecha'].dt.dayofyear
serie_interp['t_step'] = (serie_interp['fecha'] - serie_interp['fecha'].min()).dt.days
serie_interp['sen_estacional'] = np.sin(2 * np.pi * serie_interp['dia_juliano'] / 365.25)
serie_interp['cos_estacional'] = np.cos(2 * np.pi * serie_interp['dia_juliano'] / 365.25)

features = ['t_step', 'EVI', 'sen_estacional', 'cos_estacional']
X = serie_interp[features]
y = serie_interp['NDVI']

# Split cronológico sin fuga (80% train / 20% test)
split_idx = int(len(serie_interp) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

reg_multiple = CrispLinearRegressor(model_type='multiple', scale_features=True)
reg_multiple.fit(X_train, y_train)

print('Coeficientes ajustados:', reg_multiple.coefficients_)
print('Intercepto:', reg_multiple.intercept_)


## Fase 5: Evaluación (Evaluation) y Diagnóstico de Residuos
Inspección de supuestos econométricos y estadísticos: normalidad (Shapiro-Wilk) y correlación serial (Durbin-Watson).

In [ ]:
metricas = reg_multiple.evaluate(X_test, y_test)
print(json.dumps(metricas, indent=2))

# Gráfico de diagnóstico en 4 cuadrantes
reg_multiple.plot_diagnostics(X_test, y_test, '../reports/crisp_dm/diagnostico_residuos_regresion.png')
from IPython.display import Image
Image(filename='../reports/crisp_dm/diagnostico_residuos_regresion.png')


## Fase 6: Despliegue (Deployment)
Integración con los agentes de Skyfusion Analytics (OracleAgent y ReportingAgent).

In [ ]:
despliegue_file = Path('../reports/crisp_dm/alerta_despliegue_skyfusion.json')
if despliegue_file.exists():
    with open(despliegue_file, 'r', encoding='utf-8') as f:
        alerta = json.load(f)
    print(json.dumps(alerta, indent=2))
